In [ ]:
# ============================================================
# benchmark_fetal_plane_fixed.py
# Paper-style benchmark — FULLY CORRECTED VERSION
#
# Fixes applied:
#  Fix 1  — Single dataset instantiation with TransformSubset wrapper
#  Fix 2  — Per-fold seeding for reproducibility
#  Fix 3  — Checkpoint on macro-F1, not accuracy
#  Fix 4  — All metrics (F1, AUC, precision, recall) returned from CV
#  Fix 5  — CosineAnnealingLR replaces aggressive StepLR
#  Fix 6  — elif chain in EncoderClassifier.forward
#  Fix 7  — Windows paths removed; use config variables
#  Fix 8  — usfm added to AdamW group (was usfmae mismatch)
#  Fix 9  — Class-weighted CrossEntropyLoss
#  Fix 12 — patient_id fallback warns instead of silently breaking
#  Fix 13 — num_workers set to cpu_count//2
#  Fix 14 — Per-model normalization (DINOv2 uses ImageNet stats)
#  Fix 15 — pin_memory=True + persistent_workers=True
#  Fix 16 — O(N) running accuracy counter in train loop
#  Fix 17 — CSV stores mean±std for all metrics

In [ ]:
# ============================================================

import os
import re
import copy
import random
import numpy as np
from omegaconf import OmegaConf
import sys

import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

In [ ]:
# ============================================================
# 1. CONFIGURATION

In [ ]:
# ============================================================

IMAGE_ROOT    = r"/home/user/Benchmark_Modal/Image_F"

# Foundation model roots — used inside encoder constructors (Fix 7)
ULTRASAM_ROOT = r"/home/user/Benchmark_Modal/UltraSam"
MOFO_ROOT     = r"/home/user/Benchmark_Modal/MOFO"
USFM_ROOT     = r"/home/user/Benchmark_Modal/Benchmarking_Model_fetal_classification/USFM"

# Checkpoint paths
USF_MAE_CKPT  = r"/home/user/Benchmark_Modal/USF-MAE/mae_US_saved_models/USF-MAE_fold1_lr0.0003_wd0.01_epochs100.pt"
ULTRASAM_CKPT = r"/home/user/Benchmark_Modal/UltraSam.pth"
MOFO_CKPT     = r"/home/user/Benchmark_Modal/mofo_weight_sel.pth"
OUR_MODEL_CKPT = None   # set to a path to warm-start your own model

NUM_CLASSES  = 6
IMAGE_SIZE   = 224
BATCH_SIZE   = 32
EPOCHS       = 20
LR           = 1e-4
WEIGHT_DECAY = 1e-4
NUM_FOLDS    = 5

# Fix 13: use parallel workers
NUM_WORKERS  = min(4, os.cpu_count() // 2)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Workers:", NUM_WORKERS)

In [ ]:
# ============================================================
# 2. DATASET

In [ ]:
# ============================================================

def extract_patient_id(image_path):
    """
    Extract patient group key from filename.
    Fix 12: warns when pattern not found instead of silently returning
    the bare filename (which would break StratifiedGroupKFold grouping).
    """
    name = os.path.basename(image_path)
    match = re.search(r"(Patient\d+)", name, re.IGNORECASE)
    if match:
        return match.group(1)

    # Warn so the user knows grouping may be broken for this file
    print(f"Warning: no PatientXXX pattern in '{name}'. "
          f"Using filename as group — cross-val leakage guard may be weak.")
    return os.path.splitext(name)[0]


class FolderFetalPlaneDataset(Dataset):
    """
    Reads images from class-named subdirectories under image_root.
    Fix 1: __getitem__ always returns a raw PIL image so that
    TransformSubset can apply the correct split-specific transform.
    """

    CLASS_MAP = {
        "Fetal abdomen": 0,
        "Fetal brain":   1,
        "Fetal femur":   2,
        "Fetal thorax":  3,
        "Maternal cervix": 4,
        "Other":         5,
    }

    def __init__(self, image_root):
        self.image_root  = image_root
        self.class_names = list(self.CLASS_MAP.keys())
        self.class_to_idx = self.CLASS_MAP

        self.samples = []
        valid_exts   = (".png", ".jpg", ".jpeg", ".bmp")

        for class_name in self.class_names:
            class_folder = os.path.join(image_root, class_name)
            if not os.path.isdir(class_folder):
                print(f"Warning: class folder not found: {class_folder}")
                continue
            for root_dir, _, files in os.walk(class_folder):
                for fname in files:
                    if fname.lower().endswith(valid_exts):
                        path  = os.path.join(root_dir, fname)
                        label = self.class_to_idx[class_name]
                        self.samples.append((path, label, class_name))

        if not self.samples:
            print(f"Warning: no images found in {image_root}.")

        print("Classes:", self.class_to_idx)
        print("Total images:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, label, _ = self.samples[idx]
        # Return PIL image — transform is applied by TransformSubset (Fix 1)
        image = Image.open(image_path).convert("RGB")
        return image, torch.tensor(label, dtype=torch.long)


class TransformSubset(Dataset):
    """
    Fix 1: Wraps a Subset and applies a transform at __getitem__ time.
    This avoids building two full FolderFetalPlaneDataset instances per fold.
    """

    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
# ============================================================
# 3. TRANSFORMS
# Fix 14: DINOv2 needs ImageNet normalization; ultrasound models use [0.5,0.5,0.5]

In [ ]:
# ============================================================

IMAGENET_MEAN    = [0.485, 0.456, 0.406]
IMAGENET_STD     = [0.229, 0.224, 0.225]
ULTRASOUND_MEAN  = [0.5,   0.5,   0.5  ]
ULTRASOUND_STD   = [0.5,   0.5,   0.5  ]

# Models pretrained on ImageNet — need ImageNet norm for correct feature alignment
IMAGENET_MODELS  = {"dino", "resnet50", "efficientnetv2"}


def get_transforms(model_name: str):
    """Return (train_transform, val_transform) appropriate for model_name."""
    if model_name.lower() in IMAGENET_MODELS:
        mean, std = IMAGENET_MEAN, IMAGENET_STD
    else:
        mean, std = ULTRASOUND_MEAN, ULTRASOUND_STD

    train_tf = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])
    return train_tf, val_tf

In [ ]:
# ============================================================
# 4. CLASSIFICATION WRAPPER
# Fix 6: sequential `if` replaced with `elif` chain

In [ ]:
# ============================================================

class EncoderClassifier(nn.Module):
    def __init__(self, encoder, feature_dim, num_classes):
        super().__init__()
        self.encoder    = encoder
        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        features = self.encoder(x)

        # Unwrap dict outputs
        if isinstance(features, dict):
            if "pooler_output" in features:
                features = features["pooler_output"]
            elif "last_hidden_state" in features:
                features = features["last_hidden_state"][:, 0, :]
            elif "features" in features:
                features = features["features"]

        # Unwrap tuple/list outputs
        if isinstance(features, (tuple, list)):
            features = features[0]

        # Fix 6: elif so only ONE branch fires
        if features.ndim == 4:
            features = torch.mean(features, dim=[2, 3])   # CNN spatial → (B, C)
        elif features.ndim == 3:
            features = features[:, 0, :]                  # ViT CLS token → (B, D)

        return self.classifier(features)

In [ ]:
# ============================================================
# 5. BASELINE ENCODERS

In [ ]:
# ============================================================

class ResNet50Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.feature_dim = model.fc.in_features
        model.fc         = nn.Identity()
        self.encoder     = model

    def forward(self, x):
        return self.encoder(x)


class EfficientNetV2Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        model = models.efficientnet_v2_s(
            weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
        )
        self.feature_dim  = model.classifier[1].in_features
        model.classifier  = nn.Identity()
        self.encoder      = model

    def forward(self, x):
        return self.encoder(x)


class DINOEncoder(nn.Module):
    def __init__(self, model_name="facebook/dinov2-base"):
        super().__init__()
        from transformers import AutoModel
        self.encoder     = AutoModel.from_pretrained(model_name)
        self.feature_dim = self.encoder.config.hidden_size

    def forward(self, x):
        out = self.encoder(pixel_values=x)
        return out.last_hidden_state[:, 0, :]

In [ ]:
# ============================================================
# 6. FOUNDATION MODEL ENCODERS
# Fix 7: no hardcoded paths — use config variables via default args

In [ ]:
# ============================================================

class USFMAEEncoder(nn.Module):
    def __init__(self, checkpoint_path):
        super().__init__()
        import timm

        self.encoder = timm.create_model(
            "vit_base_patch16_224",
            pretrained=False,
            num_classes=0,
        )

        ckpt = torch.load(checkpoint_path, map_location="cpu")
        if isinstance(ckpt, dict):
            ckpt = ckpt.get("model", ckpt.get("state_dict", ckpt))

        clean_ckpt = {}
        for k, v in ckpt.items():
            k = k.replace("module.", "")
            k = k.replace("encoder.", "")
            k = k.replace("mae.", "")
            if k.startswith("decoder") or k.startswith("mask_token"):
                continue
            clean_ckpt[k] = v

        missing, unexpected = self.encoder.load_state_dict(clean_ckpt, strict=False)
        print(f"USF-MAE  missing={len(missing)}  unexpected={len(unexpected)}")

        self.feature_dim = 768

    def forward(self, x):
        features = self.encoder.forward_features(x)
        if features.ndim == 3:
            return features[:, 0, :]
        return features


class UltraSAMEncoder(nn.Module):
    def __init__(self, checkpoint_path, model_root=ULTRASAM_ROOT):
        super().__init__()
        # Fix 7: use config variable, not a hardcoded Windows path
        sys.path.append(model_root)
        from endosam.models.backbones.vit_sam import ViTSAMcls

        self.encoder = ViTSAMcls(
            arch="base",
            img_size=224,
            patch_size=16,
            out_indices=[11],
            use_abs_pos=True,
            use_rel_pos=True,
            window_size=14,
            out_channels=0,
        )

        ckpt = torch.load(checkpoint_path, map_location="cpu")
        if isinstance(ckpt, dict):
            ckpt = ckpt.get("state_dict", ckpt.get("model", ckpt))

        clean_ckpt = {
            k.replace("module.", "").replace("backbone.", "").replace("image_encoder.", ""): v
            for k, v in ckpt.items()
        }

        missing, unexpected = self.encoder.load_state_dict(clean_ckpt, strict=False)
        print(f"UltraSAM  missing={len(missing)}  unexpected={len(unexpected)}")

        self.feature_dim = 768

    def forward(self, x):
        _, feature_vector = self.encoder(x)
        return feature_vector


class MOFOEncoder(nn.Module):
    def __init__(self, checkpoint_path, model_root=MOFO_ROOT):
        super().__init__()
        # Fix 7: use config variable, not a hardcoded Windows path
        sys.path.append(model_root)
        from model.MOFO import backbone_encoder

        self.encoder = backbone_encoder(
            patch_size=4,
            embed_dim=64,
            depth=[2, 4, 32, 2],
            split_size=[1, 2, 7, 7],
            num_heads=[2, 4, 8, 16],
            mlp_ratio=4.0,
        )

        ckpt = torch.load(checkpoint_path, map_location="cpu")
        if isinstance(ckpt, dict):
            ckpt = ckpt.get("state_dict", ckpt.get("model", ckpt))

        clean_ckpt = {}
        for k, v in ckpt.items():
            k = k.replace("module.", "")
            for prefix in ("backbone_encoder.", "encoder.", "backbone."):
                if k.startswith(prefix):
                    k = k[len(prefix):]
                    break
            clean_ckpt[k] = v

        missing, unexpected = self.encoder.load_state_dict(clean_ckpt, strict=False)
        print(f"MOFO  missing={len(missing)}  unexpected={len(unexpected)}")

        self.feature_dim = 512

    def forward(self, x):
        features = self.encoder(x)
        if isinstance(features, (tuple, list)):
            features = features[-1]
        # Fix 6 pattern: only one branch fires
        if features.ndim == 4:
            features = torch.mean(features, dim=[2, 3])
        elif features.ndim == 3:
            features = features[:, 0, :]
        return features

In [ ]:
# ============================================================
# 7. MODEL FACTORY

In [ ]:
# ============================================================

def build_model(model_name: str, num_classes: int) -> EncoderClassifier:
    name = model_name.lower()

    if name == "resnet50":
        encoder = ResNet50Encoder()
    elif name == "efficientnetv2":
        encoder = EfficientNetV2Encoder()
    elif name == "dino":
        encoder = DINOEncoder()
    elif name in ("usfm", "usfmae"):
        encoder = USFMAEEncoder(USF_MAE_CKPT)
    elif name == "ultrasam":
        encoder = UltraSAMEncoder(ULTRASAM_CKPT)
    elif name == "mofo":
        encoder = MOFOEncoder(MOFO_CKPT)
    # elif name == "ours":
    #     encoder = OurModelEncoder()
    else:
        raise ValueError(f"Unknown model name: {model_name}")

    return EncoderClassifier(
        encoder=encoder,
        feature_dim=encoder.feature_dim,
        num_classes=num_classes,
    )

In [ ]:
# ============================================================
# 8. LINEAR PROBE / FINE-TUNE MODE + OPTIMIZER
# Fix 8: "usfm" added to AdamW group (was "usfmae" — name mismatch)

In [ ]:
# ============================================================

def set_training_mode(model: EncoderClassifier, mode: str) -> EncoderClassifier:
    if mode == "linear_probe":
        for param in model.encoder.parameters():
            param.requires_grad = False
        for param in model.classifier.parameters():
            param.requires_grad = True
    elif mode == "fine_tune":
        for param in model.parameters():
            param.requires_grad = True
    return model


def get_optimizer(model: EncoderClassifier, model_name: str):
    params = filter(lambda p: p.requires_grad, model.parameters())
    # Fix 8: "usfm" added so it actually gets AdamW
    if model_name.lower() in {"ultrasam", "dino", "usfm", "usfmae", "ours"}:
        return optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    return optim.Adam(params, lr=LR, weight_decay=WEIGHT_DECAY)

In [ ]:
# ============================================================
# 9. TRAINING
# Fix 16: O(N) running accuracy counter — no repeated accuracy_score calls

In [ ]:
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, desc="Train"):
    model.train()

    total_loss      = 0.0
    running_correct = 0      # Fix 16
    running_total   = 0      # Fix 16
    all_preds       = []
    all_labels      = []

    pbar = tqdm(loader, desc=desc, leave=False, unit="batch")

    for images, labels in pbar:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        preds = torch.argmax(logits, dim=1)

        total_loss      += loss.item() * images.size(0)
        running_correct += (preds == labels).sum().item()   # Fix 16
        running_total   += labels.size(0)                   # Fix 16
        running_acc      = running_correct / running_total  # Fix 16

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

        pbar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{running_acc:.4f}")

    epoch_acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader.dataset), epoch_acc

In [ ]:
# ============================================================
# 10. VALIDATION

In [ ]:
# ============================================================

def evaluate(model, loader, criterion, desc="Val"):
    model.eval()

    total_loss = 0.0
    all_preds  = []
    all_labels = []
    all_probs  = []

    pbar = tqdm(loader, desc=desc, leave=False, unit="batch")

    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(images)
            loss   = criterion(logits, labels)

            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs,   dim=1)

            total_loss += loss.item() * images.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

            pbar.set_postfix(loss=f"{loss.item():.4f}")

    try:
        auc = roc_auc_score(
            all_labels,
            np.array(all_probs),
            multi_class="ovr",
            average="macro",
        )
    except Exception:
        auc = 0.0

    return {
        "loss":      total_loss / len(loader.dataset),
        "acc":       accuracy_score(all_labels, all_preds),
        "precision": precision_score(all_labels, all_preds, average="macro", zero_division=0),
        "recall":    recall_score(   all_labels, all_preds, average="macro", zero_division=0),
        "f1":        f1_score(       all_labels, all_preds, average="macro", zero_division=0),
        "auc":       auc,
    }

In [ ]:
# ============================================================
# 11. TRAIN ONE FOLD
# Fix 1  — single dataset + TransformSubset
# Fix 2  — per-fold seed
# Fix 3  — checkpoint on macro-F1
# Fix 4  — return full best-metrics dict
# Fix 5  — CosineAnnealingLR
# Fix 9  — class-weighted loss
# Fix 13 — num_workers
# Fix 14 — per-model normalization via get_transforms()
# Fix 15 — pin_memory + persistent_workers

In [ ]:
# ============================================================

def train_one_fold(
    model_name: str,
    mode: str,
    train_idx,
    val_idx,
    fold_id: int,
    base_dataset: FolderFetalPlaneDataset,
):
    tqdm.write(f"\nModel: {model_name} | Mode: {mode} | Fold: {fold_id}")

    # Fix 2: seed everything for this fold
    seed = 42 + fold_id
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Fix 1: single dataset, per-split transforms
    # Fix 14: correct normalization per model
    train_transform, val_transform = get_transforms(model_name)
    train_dataset = TransformSubset(Subset(base_dataset, train_idx), train_transform)
    val_dataset   = TransformSubset(Subset(base_dataset, val_idx),   val_transform)

    # Fix 13 + Fix 15: parallel workers + pinned memory
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )

    model     = build_model(model_name, NUM_CLASSES)
    model     = set_training_mode(model, mode)
    model     = model.to(DEVICE)

    # Fix 9: class-weighted loss to handle FETAL_PLANES_DB imbalance
    train_labels  = [base_dataset.samples[i][1] for i in train_idx]
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(NUM_CLASSES),
        y=train_labels,
    )
    weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
    criterion     = nn.CrossEntropyLoss(weight=weight_tensor)

    optimizer = get_optimizer(model, model_name)

    # Fix 5: cosine schedule — LR decays smoothly over all EPOCHS
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-6
    )

    # Fix 3: track best F1, not accuracy
    best_val_f1  = 0.0
    best_metrics = {}
    best_weights = copy.deepcopy(model.state_dict())

    epoch_bar = tqdm(
        range(EPOCHS),
        desc=f"{model_name} | {mode} | fold {fold_id}",
        leave=False,
        unit="epoch",
    )

    for epoch in epoch_bar:
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer,
            desc=f"  fold{fold_id} train e{epoch+1}/{EPOCHS}",
        )
        val_metrics = evaluate(
            model, val_loader, criterion,
            desc=f"  fold{fold_id} val   e{epoch+1}/{EPOCHS}",
        )

        scheduler.step()

        epoch_bar.set_postfix(
            tr_loss=f"{train_loss:.4f}",
            tr_acc= f"{train_acc:.4f}",
            val_f1= f"{val_metrics['f1']:.4f}",
            val_auc=f"{val_metrics['auc']:.4f}",
        )
        tqdm.write(
            f"Epoch [{epoch+1}/{EPOCHS}] "
            f"Train Loss={train_loss:.4f} Acc={train_acc:.4f} | "
            f"Val Acc={val_metrics['acc']:.4f} F1={val_metrics['f1']:.4f} "
            f"AUC={val_metrics['auc']:.4f} "
            f"LR={scheduler.get_last_lr()[0]:.2e}"
        )

        # Fix 3: save best by macro-F1
        if val_metrics["f1"] > best_val_f1:
            best_val_f1  = val_metrics["f1"]
            best_metrics = val_metrics.copy()
            best_weights = copy.deepcopy(model.state_dict())

    os.makedirs("checkpoints", exist_ok=True)
    save_path = os.path.join(
        "checkpoints", f"{model_name}_{mode}_fold{fold_id}.pth"
    )
    torch.save(best_weights, save_path)
    tqdm.write(f"  Saved best checkpoint → {save_path}  (F1={best_val_f1:.4f})")

    # Fix 4: return the full metrics dict, not just accuracy
    return best_metrics, save_path

In [ ]:
# ============================================================
# 12. CROSS-VALIDATION
# Fix 4: aggregate mean ± std for every metric

In [ ]:
# ============================================================

def run_cross_validation(model_name: str, mode: str):
    # Fix 1: build the full dataset once — folds share it
    base_dataset = FolderFetalPlaneDataset(IMAGE_ROOT)

    labels = [s[1] for s in base_dataset.samples]
    groups = [extract_patient_id(s[0]) for s in base_dataset.samples]

    if len(set(groups)) >= NUM_FOLDS:
        splitter = StratifiedGroupKFold(
            n_splits=NUM_FOLDS, shuffle=True, random_state=42
        )
        splits = list(splitter.split(base_dataset.samples, labels, groups))
    else:
        print("Warning: fewer unique patient groups than folds — using StratifiedKFold.")
        splitter = StratifiedKFold(
            n_splits=NUM_FOLDS, shuffle=True, random_state=42
        )
        splits = list(splitter.split(base_dataset.samples, labels))

    fold_metrics     = []
    checkpoint_paths = []

    for fold_id, (train_idx, val_idx) in enumerate(splits, start=1):
        metrics, ckpt_path = train_one_fold(
            model_name=model_name,
            mode=mode,
            train_idx=train_idx,
            val_idx=val_idx,
            fold_id=fold_id,
            base_dataset=base_dataset,   # Fix 1: pass shared dataset
        )
        fold_metrics.append(metrics)
        checkpoint_paths.append(ckpt_path)

    # Fix 4 + Fix 17: mean ± std for every metric
    summary = {}
    for key in ("loss", "acc", "precision", "recall", "f1", "auc"):
        values = [m[key] for m in fold_metrics]
        summary[f"{key}_mean"] = float(np.mean(values))
        summary[f"{key}_std"]  = float(np.std(values))

    return summary, checkpoint_paths

In [ ]:
# ============================================================
# 13. MAIN
# Fix 17: CSV stores all metrics with mean ± std

In [ ]:
# ============================================================

def main():
    models_to_test = [
        "resnet50",
        "efficientnetv2",
        "dino",
        "usfm",
        "mofo",
        # "ultrasam",   # uncomment once checkpoint + env confirmed on Linux
        # "ours",       # uncomment once OurModelEncoder is implemented
    ]

    modes = ["linear_probe", "fine_tune"]

    results = []

    run_bar = tqdm(
        [(m, mo) for m in models_to_test for mo in modes],
        desc="Benchmark",
        unit="run",
    )

    for model_name, mode in run_bar:
        run_bar.set_description(f"Benchmark [{model_name} | {mode}]")
        tqdm.write("\n" + "=" * 80)
        tqdm.write(f"Running: {model_name}  mode={mode}")
        tqdm.write("=" * 80)

        summary, checkpoint_paths = run_cross_validation(model_name, mode)

        # Fix 17: store all metrics
        result = {
            "model": model_name,
            "mode":  mode,
            **summary,
            "checkpoints": str(checkpoint_paths),
        }
        results.append(result)

        # Write after every run so a crash doesn't lose earlier results
        pd.DataFrame(results).to_csv("benchmark_results.csv", index=False)

        # Pretty-print this run's summary
        tqdm.write(
            f"\n  Results — {model_name} [{mode}]\n"
            + "\n".join(
                f"    {k:20s}: {summary[k]:.4f}"
                for k in summary
            )
        )

    print("\n" + "=" * 80)
    print("FINAL BENCHMARK RESULTS")
    print("=" * 80)
    df = pd.DataFrame(results)
    # Display key columns
    display_cols = [
        "model", "mode",
        "acc_mean", "acc_std",
        "f1_mean",  "f1_std",
        "auc_mean", "auc_std",
        "precision_mean", "recall_mean",
    ]
    print(df[[c for c in display_cols if c in df.columns]].to_string(index=False))
    print("\nFull results saved to: benchmark_results.csv")


if __name__ == "__main__":
    main()